# Notebook 02 — Formulation QUBO d'assignation features→kernels

**Objectif :** construire et explorer le QUBO d'assignation.

## Plan
1. Calcul des alignments marginaux `a[k,m]`
2. Construction de la matrice QUBO
3. Exploration des hyperparamètres λ et μ₁
4. Résolution brute-force sur petit cas (d=8, M=2, Q=4 → 16 variables)
5. Vérification : AUC solution QUBO vs aléatoire

## Formulation

$$\min_{x \in \{0,1\}^{d \times M}} -\sum_{k,m} a_{k,m} x_{k,m} + \lambda \sum_k \sum_{m < m'} x_{k,m} x_{k,m'} + \mu_1 \sum_m \left(\sum_k x_{k,m} - Q\right)^2$$

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.data.loaders import load_breast_cancer_data, load_german_credit, subsample
from src.preprocessing.scaler import QuantumScaler
from src.kernels.analytical import KERNEL_CONFIGS
from src.qubo.assignment_qubo import (
    compute_marginal_alignments, build_qubo_matrix, energy,
    decode_assignment, check_constraints
)
from src.qubo.solvers import solve_brute_force

plt.rcParams.update({'font.family': 'sans-serif', 'figure.dpi': 130, 'savefig.bbox': 'tight'})

# Données
X_bc, y_bc, _ = load_breast_cancer_data()
X_bc_s = QuantumScaler().fit_transform(X_bc)
X_gc, y_gc, _ = load_german_credit()
X_gc, y_gc = subsample(X_gc, y_gc, n=100, seed=42)
X_gc_s = QuantumScaler().fit_transform(X_gc)

print('Setup OK')
print(f'BC: {X_bc_s.shape},  GC: {X_gc_s.shape}')

## 1. Calcul des alignments marginaux `a[k,m]`

Pour chaque paire (feature k, kernel m) : quel est l'alignment centré
du kernel m en utilisant seulement la feature k (+ padding) ?

In [ ]:
Q = 4
M = 5
kernel_configs = [
    ('Z',  0.5), ('ZZ', 1.0), ('XZ', 0.5), ('ZZ', 2.0), ('Z', 2.0)
]

print('Calcul des alignments marginaux BC (d=30, M=5)...')
a_bc = compute_marginal_alignments(X_bc_s[:100], y_bc[:100], M, Q, kernel_configs, padding='top')
print(f'a_bc shape: {a_bc.shape},  range: [{a_bc.min():.4f}, {a_bc.max():.4f}]')

print('\nCalcul des alignments marginaux GC (d=20, M=5)...')
a_gc = compute_marginal_alignments(X_gc_s, y_gc, M, Q, kernel_configs, padding='top')
print(f'a_gc shape: {a_gc.shape},  range: [{a_gc.min():.4f}, {a_gc.max():.4f}]')

In [ ]:
# Visualisation de a[k,m]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, a, title in zip(axes, [a_bc, a_gc], ['Breast Cancer (d=30)', 'German Credit (d=20)']):
    im = ax.imshow(a, aspect='auto', cmap='RdYlGn')
    ax.set_xlabel('Kernel m')
    ax.set_ylabel('Feature k')
    ax.set_title(f'Alignment marginal a[k,m]\n{title}')
    ax.set_xticks(range(M))
    ax.set_xticklabels([f'K{m}\n({cfg[0]} α={cfg[1]})' for m, cfg in enumerate(kernel_configs)], fontsize=8)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('../results/figures/02_marginal_alignments.png')
plt.show()
print('→ Les features n\'ont pas le même alignment selon le kernel : structure à exploiter.')

## 2. Construction de la matrice QUBO

In [ ]:
# Paramètres QUBO
lambda_div = 0.5
mu1        = 2.0

Q_mat_gc = build_qubo_matrix(a_gc, d=X_gc_s.shape[1], M=M, Q=Q,
                               lambda_div=lambda_div, mu1=mu1)

print(f'Matrice QUBO GC : {Q_mat_gc.shape}')
print(f'  Valeurs diag   : [{np.diag(Q_mat_gc).min():.3f}, {np.diag(Q_mat_gc).max():.3f}]')
print(f'  Valeurs offdiag: [{Q_mat_gc[Q_mat_gc > 0].min():.3f}, {Q_mat_gc.max():.3f}]')

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(Q_mat_gc, cmap='RdBu_r', vmin=-abs(Q_mat_gc).max(), vmax=abs(Q_mat_gc).max())
ax.set_title(f'Matrice QUBO — German Credit\n(d=20, M={M}, λ={lambda_div}, μ₁={mu1})\nVariables: {Q_mat_gc.shape[0]} = d×M = 20×5')
ax.set_xlabel('Variable i = k·M + m')
ax.set_ylabel('Variable j = k\'·M + m\'')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('../results/figures/02_qubo_matrix_gc.png')
plt.show()

## 3. Résolution brute-force — cas pédagogique (d=8, M=2, Q=4 → 16 variables)

In [ ]:
d_small, M_small, Q_small = 8, 2, 4
kc_small = kernel_configs[:M_small]

X_small = X_bc_s[:100, :d_small]
y_small = y_bc[:100]

a_small = compute_marginal_alignments(X_small, y_small, M_small, Q_small, kc_small, padding='top')
Q_mat_small = build_qubo_matrix(a_small, d_small, M_small, Q_small,
                                  lambda_div=lambda_div, mu1=mu1)

print(f'Cas pédagogique : d={d_small}, M={M_small}, Q={Q_small}')
print(f'Variables QUBO : {d_small * M_small} → brute force 2^{d_small * M_small} = {2**(d_small*M_small):,}')
print('\nRésolution brute-force...')

result_bf = solve_brute_force(Q_mat_small, d_small, M_small, Q_small)
assignment_bf = decode_assignment(result_bf['x'], d_small, M_small, Q_small)
valid, viol = check_constraints(assignment_bf, d_small, M_small, Q_small)

print(f'\nSolution optimale (brute-force):')
print(f'  Énergie QUBO : {result_bf["energy"]:.4f}')
print(f'  Assignation  : {assignment_bf}')
print(f'  Valide       : {valid}')

## 4. Balayage de λ (diversity penalty)

In [ ]:
from src.qubo.solvers import solve_simulated_annealing
from src.kernels.subset_kernels import build_subset_kernels_train_test
from src.mkl.combiner import MultipleKernelCombiner
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

lambdas = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]
aucs_per_lambda = []

for lam in lambdas:
    Q_mat = build_qubo_matrix(a_gc, d=X_gc_s.shape[1], M=M, Q=Q,
                               lambda_div=lam, mu1=mu1)
    res = solve_simulated_annealing(Q_mat, X_gc_s.shape[1], M, Q,
                                      n_iter=20_000, seed=42)
    asgn = decode_assignment(res['x'], X_gc_s.shape[1], M, Q)

    # Quick AUC estimate (3-fold only)
    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    fold_aucs = []
    for tr, te in kf.split(X_gc_s, y_gc):
        K_trs, K_tes = build_subset_kernels_train_test(
            X_gc_s[tr], X_gc_s[te], asgn, kernel_configs
        )
        cb = MultipleKernelCombiner(method='centered')
        K_tr_c = cb.fit_combine(K_trs, y_gc[tr])
        K_te_c = cb.combine(K_tes)
        svm = SVC(kernel='precomputed', C=1.0, probability=True)
        svm.fit(K_tr_c, y_gc[tr])
        fold_aucs.append(roc_auc_score(y_gc[te], svm.predict_proba(K_te_c)[:, 1]))

    aucs_per_lambda.append(np.mean(fold_aucs))
    print(f'  λ={lam:.2f}  AUC={np.mean(fold_aucs):.3f}  E={res["energy"]:.3f}')

best_lambda = lambdas[np.argmax(aucs_per_lambda)]
print(f'\n→ Meilleur λ = {best_lambda}')

plt.figure(figsize=(6, 4))
plt.plot(lambdas, aucs_per_lambda, 'o-', color='#2ecc71', lw=2)
plt.axvline(best_lambda, ls='--', color='gray')
plt.xlabel('λ (diversity penalty)')
plt.ylabel('AUC (3-fold CV, German Credit)')
plt.title('Sensibilité à λ — QUBO Assignation')
plt.tight_layout()
plt.savefig('../results/figures/02_lambda_sweep.png')
plt.show()

## Conclusion

- La matrice QUBO encode bien le trade-off qualité / diversité
- λ optimal ≈ 0.5 pour German Credit
- La solution brute-force est valide (Q features par kernel)

**Suite → Notebook 03 : comparaison SA vs QAOA**